# Local Raw Data Demo

This notebook starts from the raw Kaggle files only: `data/train.csv` and `data/test.csv`. It recomputes the key dataset and EDA numbers used in the presentation. It does not read precomputed experiment tables.

In [ ]:
from pathlib import Path

def find_project_root(start: Path = Path.cwd()) -> Path:
    for path in [start, *start.parents]:
        if (path / "data" / "train.csv").exists() and (path / "data" / "test.csv").exists():
            return path
    raise FileNotFoundError("Open this notebook from inside the repository folder.")

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"

In [ ]:
import pandas as pd

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")

train.head()

## Dataset Summary

In [ ]:
group_id = train["PassengerId"].str[:4]
group_size = group_id.value_counts()
group_labels = train.assign(GroupId=group_id).groupby("GroupId")["Transported"]

dataset_summary = pd.DataFrame([
    ["Training rows", len(train)],
    ["Test rows", len(test)],
    ["Raw predictive columns", train.drop(columns=["Transported"]).shape[1]],
    ["Positive class rate", f"{train['Transported'].mean() * 100:.2f}%"],
    ["Largest missing rate", f"{train.isna().mean().max() * 100:.2f}%"],
    ["Multi-passenger row rate", f"{group_id.map(group_size).gt(1).mean() * 100:.1f}%"],
    ["Group-homogeneous rate", f"{group_labels.nunique().eq(1).mean() * 100:.2f}%"],
], columns=["Metric", "Value"])

dataset_summary

## Missing Values

In [ ]:
missing_rate = (
    train.isna()
    .mean()
    .sort_values(ascending=False)
    .mul(100)
    .round(2)
    .rename("Missing rate (%)")
    .reset_index()
    .rename(columns={"index": "Column"})
)

missing_rate.head(10)

## Cabin, HomePlanet, Spend, And Group Signals

In [ ]:
eda = train.copy()
cabin = eda["Cabin"].str.split("/", expand=True)
eda["Deck"] = cabin[0]
eda["CabinSide"] = cabin[2]

spend_cols = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
eda["TotalSpend"] = eda[spend_cols].fillna(0).sum(axis=1)
eda["NoSpend"] = eda["TotalSpend"].eq(0)

eda_signals = pd.DataFrame([
    ["Deck B transport rate", f"{eda.groupby('Deck')['Transported'].mean().loc['B'] * 100:.0f}%"],
    ["Deck T transport rate", f"{eda.groupby('Deck')['Transported'].mean().loc['T'] * 100:.0f}%"],
    ["Europa transport rate", f"{eda.groupby('HomePlanet')['Transported'].mean().loc['Europa'] * 100:.0f}%"],
    ["Earth transport rate", f"{eda.groupby('HomePlanet')['Transported'].mean().loc['Earth'] * 100:.0f}%"],
    ["No-spend rate if transported", f"{eda.groupby('Transported')['NoSpend'].mean().loc[True] * 100:.1f}%"],
    ["No-spend rate if not transported", f"{eda.groupby('Transported')['NoSpend'].mean().loc[False] * 100:.1f}%"],
], columns=["Signal", "Value"])

eda_signals

In [ ]:
deck_rates = eda.groupby("Deck")["Transported"].mean().sort_values(ascending=False).mul(100).round(1)
homeplanet_rates = eda.groupby("HomePlanet")["Transported"].mean().sort_values(ascending=False).mul(100).round(1)
group_distribution = group_id.map(group_size).value_counts().sort_index()

deck_rates.rename("Transport rate (%)").reset_index()

In [ ]:
homeplanet_rates.rename("Transport rate (%)").reset_index()

In [ ]:
group_distribution.rename("Passenger rows").reset_index().rename(columns={"index": "Group size"})

This notebook is raw-data-only. The Kaggle public-best training demo is in `demo_081716_xgb.ipynb`; heavier local validation reproduction code is in `code/local_validation_reproduction.py`.